In [1]:
!pip install sentence-transformers faiss-cpu pandas numpy groq fastapi uvicorn sqlalchemy rouge-score nltk scikit-learn

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import re
import html
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import faiss

In [3]:


# ── 1. Load dataset ──────────────────────────────────────────────
df = pd.read_csv(
    '/content/drive/MyDrive/task/Natural-Questions-Filtered.csv',
    engine='python',
    on_bad_lines='skip'
)
print(f"Loaded {len(df)} rows")
print(df.head(3))

Loaded 86214 rows
                                            question  \
0  which is the most common use of opt-in e-mail ...   
1            how i.met your mother who is the mother   
2                   who had the most wins in the nfl   

                                        long_answers  \
0  A common example of permission marketing is a ...   
1  Tracy McConnell, better known as `` The Mother...   
2  Active quarterback Tom Brady holds the records...   

                                       short_answers Unnamed: 3 Unnamed: 4  \
0  A newsletter sent to an advertising firm's cus...        NaN        NaN   
1                                    Tracy McConnell        NaN        NaN   
2                                          Tom Brady        NaN        NaN   

  Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9  ... Unnamed: 65  \
0        NaN        NaN        NaN        NaN        NaN  ...         NaN   
1        NaN        NaN        NaN        NaN        NaN  ...    

In [4]:

# Keep only needed columns
df = df[['question', 'long_answers', 'short_answers']]

print(f"Loaded {len(df)} rows")
print(df.head(3))

Loaded 86214 rows
                                            question  \
0  which is the most common use of opt-in e-mail ...   
1            how i.met your mother who is the mother   
2                   who had the most wins in the nfl   

                                        long_answers  \
0  A common example of permission marketing is a ...   
1  Tracy McConnell, better known as `` The Mother...   
2  Active quarterback Tom Brady holds the records...   

                                       short_answers  
0  A newsletter sent to an advertising firm's cus...  
1                                    Tracy McConnell  
2                                          Tom Brady  


In [5]:
print(df.shape)

(86214, 3)


In [6]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'[^\w\s.,!?;:\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

df['question']      = df['question'].apply(clean_text)
df['long_answers']  = df['long_answers'].apply(clean_text)
df['short_answers'] = df['short_answers'].apply(clean_text)

print("✅ Text cleaned")
print(df.head(3))

✅ Text cleaned
                                            question  \
0  which is the most common use of opt-in e-mail ...   
1            how i.met your mother who is the mother   
2                   who had the most wins in the nfl   

                                        long_answers  \
0  A common example of permission marketing is a ...   
1  Tracy McConnell, better known as The Mother ''...   
2  Active quarterback Tom Brady holds the records...   

                                       short_answers  
0  A newsletter sent to an advertising firm's cus...  
1                                    Tracy McConnell  
2                                          Tom Brady  


In [7]:
def get_question_type(q):
    q = q.lower().strip()
    if q.startswith('who'):                              return 'person'
    if q.startswith('where'):                            return 'location'
    if q.startswith('when'):                             return 'time'
    if q.startswith('how many') or q.startswith('how much'): return 'quantity'
    if q.startswith('how'):                              return 'method'
    if q.startswith('what'):                             return 'definition'
    if q.startswith('why'):                              return 'reason'
    if q.startswith('which'):                            return 'selection'
    return 'other'

def get_answer_type(short):
    length = len(str(short))
    if length <= 20:  return 'factoid'
    if length <= 100: return 'phrase'
    return 'descriptive'

def get_difficulty(long):
    length = len(str(long))
    if length < 200:  return 'easy'
    if length < 600:  return 'medium'
    return 'hard'

df['question_type'] = df['question'].apply(get_question_type)
df['answer_type']   = df['short_answers'].apply(get_answer_type)
df['difficulty']    = df['long_answers'].apply(get_difficulty)

print("✅ Metadata added")
print(df[['question', 'question_type', 'answer_type', 'difficulty']].head(5))

✅ Metadata added
                                            question question_type  \
0  which is the most common use of opt-in e-mail ...     selection   
1            how i.met your mother who is the mother        method   
2                   who had the most wins in the nfl        person   
3        who played mantis guardians of the galaxy 2        person   
4  the nashville sound brought a polished and cos...         other   

   answer_type difficulty  
0       phrase     medium  
1      factoid     medium  
2      factoid     medium  
3      factoid     medium  
4  descriptive       hard  


In [8]:
print("Question types:\n", df['question_type'].value_counts())
print("\nAnswer types:\n",  df['answer_type'].value_counts())
print("\nDifficulty:\n",    df['difficulty'].value_counts())

df.to_csv('nq_processed.csv', index=False)
print("\n✅ Saved as nq_processed.csv — Shape:", df.shape)

Question types:
 question_type
person        30394
time          15886
definition    13046
location      11130
other          9774
quantity       2763
selection      1500
method         1199
reason          522
Name: count, dtype: int64

Answer types:
 answer_type
factoid        54896
phrase         27171
descriptive     4147
Name: count, dtype: int64

Difficulty:
 difficulty
medium    45595
hard      30727
easy       9892
Name: count, dtype: int64

✅ Saved as nq_processed.csv — Shape: (86214, 6)


In [9]:
def chunk_text(text, chunk_size=250, overlap=50):
    """
    Split text into word-based chunks with overlap.
    chunk_size : max words per chunk
    overlap    : words shared between consecutive chunks (preserves context)
    """
    words = text.split()

    if len(words) <= chunk_size:
        return [text]   # short enough — keep as one chunk

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap   # slide forward with overlap

    return chunks

In [10]:
rows = []

for _, row in df.iterrows():
    chunks = chunk_text(row['long_answers'])

    for i, chunk in enumerate(chunks):
        rows.append({
            'question'      : row['question'],
            'chunk'         : chunk,
            'chunk_index'   : i,
            'total_chunks'  : len(chunks),
            'short_answer'  : row['short_answers'],
            'question_type' : row['question_type'],
            'answer_type'   : row['answer_type'],
            'difficulty'    : row['difficulty'],
        })

df_chunks = pd.DataFrame(rows)

print(f"✅ Original rows : {len(df)}")
print(f"✅ After chunking: {len(df_chunks)}")
print(f"✅ Avg chunks per answer: {len(df_chunks)/len(df):.2f}")
print()
print(df_chunks.head(15))

✅ Original rows : 86214
✅ After chunking: 88220
✅ Avg chunks per answer: 1.02

                                             question  \
0   which is the most common use of opt-in e-mail ...   
1             how i.met your mother who is the mother   
2                    who had the most wins in the nfl   
3         who played mantis guardians of the galaxy 2   
4   the nashville sound brought a polished and cos...   
5     who needs to be in the car with a permit driver   
6   god's not dead a light in the darkness release...   
7   who is the current president of un general ass...   
8        when do they pull the powerball numbers 2016   
9       what is the name of the sea surrounding dubai   
10        when did the new maze runner movie come out   
11            how many players on a box lacrosse team   
12         where are the upcoming olympics to be held   
13               when did the nba 3 second rule start   
14              who plays norman bates in the tv show   

        

In [11]:
# Check chunk word lengths
df_chunks['chunk_word_count'] = df_chunks['chunk'].apply(lambda x: len(x.split()))

print("Chunk word count stats:")
print(df_chunks['chunk_word_count'].describe())

# Save
df_chunks.to_csv('nq_chunks.csv', index=False)
print("\n✅ Saved as nq_chunks.csv — Shape:", df_chunks.shape)

Chunk word count stats:
count    88220.000000
mean        92.733303
std         53.415667
min          0.000000
25%         55.000000
50%         84.000000
75%        120.000000
max        250.000000
Name: chunk_word_count, dtype: float64

✅ Saved as nq_chunks.csv — Shape: (88220, 9)


In [12]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("✅ SentenceTransformer loaded")
print("✅ CrossEncoder reranker loaded")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SentenceTransformer loaded
✅ CrossEncoder reranker loaded
Embedding dimension: 384


/tmp/ipykernel_7501/3593144553.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


In [13]:


chunks = df_chunks['chunk'].tolist()

print(f"Embedding {len(chunks)} chunks... this may take 5-10 mins on GPU")

embeddings = model.encode(
    chunks,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # cosine similarity becomes dot product — faster in FAISS
)

print(f"✅ Embeddings shape: {embeddings.shape}")

# Save embeddings so you don't have to re-run
np.save('embeddings.npy', embeddings)
print("✅ Saved as embeddings.npy")

Embedding 88220 chunks... this may take 5-10 mins on GPU


Batches:   0%|          | 0/345 [00:00<?, ?it/s]

✅ Embeddings shape: (88220, 384)
✅ Saved as embeddings.npy


In [14]:

dim = embeddings.shape[1]   # 384 for this model

# IndexFlatIP = Inner Product (cosine sim since embeddings are normalized)
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"✅ FAISS index built")
print(f"Total vectors indexed: {index.ntotal}")

# Save index
faiss.write_index(index, 'nq_faiss.index')
print("✅ Saved as nq_faiss.index")

✅ FAISS index built
Total vectors indexed: 88220
✅ Saved as nq_faiss.index


In [15]:
def search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, indices = index.search(query_embedding, top_k)

    results = df_chunks.iloc[indices[0]].copy()
    results['score'] = scores[0]

    return results[['question', 'chunk', 'short_answer', 'score']]

# Test it!
results = search("who invented the telephone?")
print(results.to_string())

                                                question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      chunk                                        short_answer     score
83729        who invented the telephone and in what year         Alexander Graham Bell is the inventor of the first practical telephone. The classic story of him saying Watson, come here ! I want to see you ! '' is a well - known part of the history of the telephone. This showed that the telephone worked, but

In [16]:



def normalize_query(query):
    query = query.strip()
    query = re.sub(r'\s+', ' ', query)
    query = query.lower()
    if not query.endswith('?') and any(
        query.startswith(w) for w in ['who','what','where','when','why','how','which']
    ):
        query = query + '?'
    return query

print("✅ normalize_query ready")

# Test
print(normalize_query("  who invented   the telephone "))
print(normalize_query("WHAT IS THE CAPITAL OF FRANCE"))

✅ normalize_query ready
who invented the telephone?
what is the capital of france?


In [17]:
def expand_query(query):
    variants = [query]
    q = query.lower()

    if q.startswith('who invented'):
        variants.append(q.replace('who invented', 'who created'))
        variants.append(q.replace('who invented', 'who is the inventor of'))
    if q.startswith('what is'):
        variants.append(q.replace('what is', 'define'))
        variants.append(q.replace('what is', 'explain'))
    if q.startswith('where is'):
        variants.append(q.replace('where is', 'location of'))
    if q.startswith('when did') or q.startswith('when was'):
        variants.append(q.replace('when did', 'what year did'))
        variants.append(q.replace('when was', 'what year was'))
    if q.startswith('how many'):
        variants.append(q.replace('how many', 'what is the number of'))

    return variants

print("✅ expand_query ready")

# Test
print(expand_query("who invented the telephone?"))
print(expand_query("what is photosynthesis?"))

✅ expand_query ready
['who invented the telephone?', 'who created the telephone?', 'who is the inventor of the telephone?']
['what is photosynthesis?', 'define photosynthesis?', 'explain photosynthesis?']


In [18]:
import re

class ConversationContext:
    def __init__(self, max_history=5):
        self.history = []
        self.max_history = max_history

    def add(self, query, answer):
        self.history.append({'query': query, 'answer': answer})
        if len(self.history) > self.max_history:
            self.history.pop(0)

    def enrich_query(self, query):
        if not self.history:
            return query

        followup_triggers = ['that', 'this', 'it', 'he', 'she', 'they',
                             'when did', 'what year', 'where did']
        is_followup = any(trigger in query.lower() for trigger in followup_triggers)

        if is_followup:
            last = self.history[-1]
            merged = f"{last['query']} {last['answer']}. {query}"

            # ✅ Removed all print statements — silent merging
            year_triggers   = ['what year', 'when did', 'when was']
            person_triggers = ['who ', 'who is', 'who was']

            if any(t in query.lower() for t in year_triggers):
                years = re.findall(r'\b(1[0-9]{3}|20[0-9]{2})\b', last['answer'])
                if years:
                    return f"DIRECT_ANSWER: {last['answer']}"

            if any(t in query.lower() for t in person_triggers):
                if len(last['answer'].split()) <= 10:
                    return f"DIRECT_ANSWER: {last['answer']}"

        return query

    def get_history(self):
        return self.history

conversation = ConversationContext()
print("✅ ConversationContext ready")

✅ ConversationContext ready


In [19]:

def pipeline_search(query, top_k=10, use_expansion=True, conversation=None):
    # Step 1: normalize
    query = normalize_query(query)

    # Step 2: enrich with conversation context
    if conversation:
        query = conversation.enrich_query(query)

    # ✅ Step 3: if answer already known from context, skip FAISS
    if isinstance(query, str) and query.startswith("DIRECT_ANSWER:"):
        print("⚡ Skipping FAISS — answer found in conversation history")
        return query.replace("DIRECT_ANSWER:", "").strip()

    # Step 4: expand query
    queries = expand_query(query) if use_expansion else [query]

    # Step 5: embed all variants and average
    query_embeddings = model.encode(
        queries,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    combined_embedding = np.mean(query_embeddings, axis=0, keepdims=True)

    # Step 6: search FAISS
    scores, indices = index.search(combined_embedding, top_k)

    results = df_chunks.iloc[indices[0]].copy()
    results['score'] = scores[0]

    return results[['question', 'chunk', 'short_answer', 'score']]

print("✅ pipeline_search ready")

# Test the full pipeline
results = pipeline_search("how many bones are in the human body?")
print(results.to_string())

✅ pipeline_search ready
                                                               question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [20]:
def rerank_results(query, results, top_k=3):
    pairs = [[query, row['chunk']] for _, row in results.iterrows()]
    scores = reranker.predict(pairs)
    results = results.copy()
    results['rerank_score'] = scores
    results = results.sort_values('rerank_score', ascending=False).head(top_k)
    return results

print("✅ rerank_results ready")

✅ rerank_results ready


In [ ]:
!pip install groq -q

from groq import Groq

GROQ_API_KEY = "add api"  # paste your key
client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client ready")

✅ Groq client ready


In [22]:
detail_triggers = [
    'in details', 'in detail', 'explain', 'elaborate', 'tell me more',
    'give me more', 'more info', 'more information', 'describe',
    'full answer', 'full explanation', 'everything about', 'all about',
    'comprehensive', 'detailed', 'thoroughly', 'step by step'
]

def wants_detail(query):
    q = query.lower()
    return any(trigger in q for trigger in detail_triggers)

print("✅ Detail detector ready")

✅ Detail detector ready


In [23]:
def build_prompt(query, retrieved_chunks, detailed=False):
    context_parts = []
    for i, row in retrieved_chunks.iterrows():
        context_parts.append(f"[Source {len(context_parts)+1}]\n{row['chunk']}")

    context = "\n\n".join(context_parts)

    if detailed:
        prompt = f"""You are a knowledgeable assistant. The user wants a comprehensive answer.
Follow these rules:
1. Answer ONLY from the context below — never use outside knowledge
2. Give a thorough, detailed explanation covering all relevant facts in the context
3. Include names, dates, numbers, and background information
4. Use full sentences and paragraphs — do not use bullet points
5. If the context doesn't contain the answer, say exactly: "Not found in context"

CONTEXT:
{context}

QUESTION: {query}

DETAILED ANSWER:"""

    else:
        prompt = f"""You are a precise Q&A assistant. Follow these rules strictly:
1. Answer ONLY from the context below — never use outside knowledge
2. Answer in the SHORTEST possible way — only what was directly asked
3. If asked "who" — return only the name, nothing else
4. If asked "what year" or "when" — return only the year, nothing else
5. If asked "where" — return only the place, nothing else
6. Never add extra facts, dates, or explanations unless directly asked
7. If the context doesn't contain the answer, say exactly: "Not found in context"

CONTEXT:
{context}

QUESTION: {query}

ANSWER (be as short as possible):"""

    return prompt

print("✅ build_prompt ready")

✅ build_prompt ready


In [24]:
def generate_answer(query, top_k=10, conversation=None):
    # Step 1: retrieve chunks
    retrieved = pipeline_search(query, top_k=top_k, conversation=conversation)

    # Step 2: direct answer from context
    if isinstance(retrieved, str):
        answer = retrieved
        if conversation:
            conversation.add(query, answer)
        return {
            "query"  : query,
            "answer" : f"(From previous context) {answer}",
            "sources": []
        }

    # Step 3: rerank
    retrieved = rerank_results(query, retrieved, top_k=3)

    # ✅ Step 4: detect if user wants detail
    detailed = wants_detail(query)
    if detailed:
        print("📖 Detailed answer mode ON")

    # Step 5: build prompt
    prompt = build_prompt(query, retrieved, detailed=detailed)

    # Step 6: call Groq
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=1024 if detailed else 512,  # more tokens for detailed
        )
        answer = response.choices[0].message.content.strip()
    except Exception as e:
        answer = f"LLM error: {str(e)}"

    if conversation:
        conversation.add(query, answer)

    return {
        "query"  : query,
        "answer" : answer,
        "sources": retrieved[['question', 'short_answer', 'rerank_score']].to_dict('records')
    }

print("✅ generate_answer ready")

✅ generate_answer ready


In [53]:
conversation = ConversationContext()

# Detailed answer
r2 = generate_answer("who invented the telephone? in details", conversation=conversation)
print("Q:", r2['query'])
print("A:", r2['answer'])

# Short answer
r1 = generate_answer("who invented the telephone?", conversation=conversation)
print("Q:", r1['query'])
print("A:", r1['answer'])
print()



📖 Detailed answer mode ON
Q: who invented the telephone? in details
A: The invention of the telephone is attributed to Alexander Graham Bell, a renowned inventor and businessman. According to historical records, Bell was the first to obtain a patent for an apparatus for transmitting vocal or other sounds telegraphically in 1876. This marked a significant milestone in the development of the telephone.

Before Bell's invention, there were various primitive sound transmitters and receivers that he experimented with. His breakthrough came when he successfully demonstrated the telephone's functionality by saying "Watson, come here! I want to see you!" This iconic phrase showcased the telephone's ability to transmit sound over a short range.

Bell's invention of the telephone was a culmination of his extensive research and experimentation in the field of sound transmission. His work laid the foundation for the development of modern telecommunications, revolutionizing the way people communica

In [25]:
conversation = ConversationContext()

# Detailed answer
r2 = generate_answer("who invented the telephone? in details", conversation=conversation)
print("Q:", r2['query'])
print("A:", r2['answer'])

# Short answer
r1 = generate_answer("who invented the telephone?", conversation=conversation)
print("Q:", r1['query'])
print("A:", r1['answer'])
print()



📖 Detailed answer mode ON
Q: who invented the telephone? in details
A: The invention of the telephone is attributed to Alexander Graham Bell, a renowned inventor and businessman. According to historical records, Bell was the first to obtain a patent for an apparatus for transmitting vocal or other sounds telegraphically in 1876. This marked a significant milestone in the development of the telephone.

Before obtaining the patent, Bell had been experimenting with various primitive sound transmitters and receivers. His work on the telephone was a culmination of his efforts to improve communication methods, particularly for the deaf and hard of hearing. Bell's invention of the telephone revolutionized the way people communicate, enabling real-time voice conversations over long distances.

The classic story of Bell's first successful telephone call is well-documented. He famously exclaimed, "Watson, come here! I want to see you!" to his assistant, Thomas Watson, demonstrating the telephone

In [26]:
result2 = generate_answer("when he invented the telephone?", conversation=conversation)
print("Q:", result2['query'])
print("A:", result2['answer'])
if result2['sources']:
    print("\nTop sources:")
    for s in result2['sources']:
        print(f"  - {s['question']} (rerank score: {s['rerank_score']:.3f})")
else:
    print("\n⚡ Answered directly from conversation history — no FAISS search needed")

Q: when he invented the telephone?
A: 1876

Top sources:
  - who invented the telephone and in what year (rerank score: 7.820)
  - who made the first telephone in the world (rerank score: 7.647)
  - who made the first mobile phone in the world (rerank score: 3.229)


In [27]:
import time
import hashlib

# Simple in-memory cache
query_cache = {}

def get_cache_key(query):
    """Create a unique key for each query."""
    return hashlib.md5(query.lower().strip().encode()).hexdigest()

def cached_generate_answer(query, top_k=10, conversation=None):
    cache_key = get_cache_key(query)

    # ✅ Check cache first
    if cache_key in query_cache:
        print(f"⚡ Cache HIT for: '{query}'")
        return query_cache[cache_key]

    print(f"🔍 Cache MISS — running full pipeline for: '{query}'")

    # Run full pipeline
    start_time = time.time()
    result = generate_answer(query, top_k=top_k, conversation=conversation)
    elapsed = time.time() - start_time

    # Add timing to result
    result['latency_seconds'] = round(elapsed, 3)

    # Save to cache
    query_cache[cache_key] = result
    print(f"✅ Cached in {elapsed:.3f}s")

    return result

print("✅ Cache system ready")

✅ Cache system ready


In [28]:
performance_log = []

def tracked_generate_answer(query, top_k=10, conversation=None):
    start = time.time()
    result = cached_generate_answer(query, top_k=top_k, conversation=conversation)
    elapsed = time.time() - start

    # Log this query
    performance_log.append({
        'query'          : query,
        'answer'         : result['answer'],
        'latency'        : round(elapsed, 3),
        'cache_hit'      : elapsed < 0.05,   # very fast = came from cache
        'sources_count'  : len(result.get('sources', [])),
        'timestamp'      : time.strftime('%Y-%m-%d %H:%M:%S')
    })

    return result

def show_performance_report():
    if not performance_log:
        print("No queries logged yet.")
        return

    df_perf = pd.DataFrame(performance_log)

    print("=" * 50)
    print("📊 PERFORMANCE REPORT")
    print("=" * 50)
    print(f"Total queries     : {len(df_perf)}")
    print(f"Cache hits        : {df_perf['cache_hit'].sum()} ({df_perf['cache_hit'].mean()*100:.1f}%)")
    print(f"Avg latency       : {df_perf['latency'].mean():.3f}s")
    print(f"Min latency       : {df_perf['latency'].min():.3f}s")
    print(f"Max latency       : {df_perf['latency'].max():.3f}s")
    print("=" * 50)
    print("\nPer-query breakdown:")
    print(df_perf[['query','latency','cache_hit','sources_count']].to_string(index=False))

print("✅ Performance tracker ready")

✅ Performance tracker ready


In [29]:
def batch_generate_answers(queries, top_k=10):
    """Process multiple queries efficiently."""
    print(f"Processing {len(queries)} queries in batch...\n")
    results = []

    for i, query in enumerate(queries):
        print(f"[{i+1}/{len(queries)}] {query}")
        result = tracked_generate_answer(query, top_k=top_k)
        results.append(result)

    print("\n✅ Batch complete")
    return results

print("✅ Batch processor ready")

✅ Batch processor ready


In [30]:
# Test batch processing
test_queries = [
    "who invented the telephone?",
    "what is the capital of france?",
    "who wrote harry potter?",
    "who invented the telephone?",   # duplicate — should hit cache
    "when was the eiffel tower built?",
]

conversation = ConversationContext()
batch_results = batch_generate_answers(test_queries)

print()
for r in batch_results:
    print(f"Q: {r['query']}")
    print(f"A: {r['answer']}\n")

Processing 5 queries in batch...

[1/5] who invented the telephone?
🔍 Cache MISS — running full pipeline for: 'who invented the telephone?'
✅ Cached in 0.549s
[2/5] what is the capital of france?
🔍 Cache MISS — running full pipeline for: 'what is the capital of france?'
✅ Cached in 3.417s
[3/5] who wrote harry potter?
🔍 Cache MISS — running full pipeline for: 'who wrote harry potter?'
✅ Cached in 1.575s
[4/5] who invented the telephone?
⚡ Cache HIT for: 'who invented the telephone?'
[5/5] when was the eiffel tower built?
🔍 Cache MISS — running full pipeline for: 'when was the eiffel tower built?'
✅ Cached in 13.710s

✅ Batch complete

Q: who invented the telephone?
A: Alexander Graham Bell

Q: what is the capital of france?
A: Paris

Q: who wrote harry potter?
A: J.K. Rowling

Q: who invented the telephone?
A: Alexander Graham Bell

Q: when was the eiffel tower built?
A: Not found in context



In [31]:
show_performance_report()

📊 PERFORMANCE REPORT
Total queries     : 5
Cache hits        : 1 (20.0%)
Avg latency       : 3.850s
Min latency       : 0.000s
Max latency       : 13.710s

Per-query breakdown:
                           query  latency  cache_hit  sources_count
     who invented the telephone?    0.549      False              3
  what is the capital of france?    3.418      False              3
         who wrote harry potter?    1.575      False              3
     who invented the telephone?    0.000       True              3
when was the eiffel tower built?   13.710      False              3
